# Langfuse Demo — Chapter 2: RAG Pipeline with Hierarchical Tracing

> **Estimated time**: ~10 minutes

---

## Scenario: Internal Documentation Assistant

The team has internal documents spread around: onboarding guides, system architecture, APIs, MLOps, data governance.  
We want an **assistant that answers technical questions by searching these docs**.

### The problem with RAG without observability

When the assistant answers incorrectly, **where is the problem?**

```
Question → [Retrieval] → [Reranking] → [LLM] → Wrong answer
               ↑              ↑           ↑
           Retrieved      Reordered   Hallucinated?
           right docs?    correctly?      
```

With Langfuse, each step becomes a **span** with its own inputs/outputs.  
You pinpoint exactly where the pipeline failed.

### Span hierarchy we'll create

```
Trace: rag-docs-assistant
└── Span: rag-pipeline
      ├── Span: retrieval          ← which docs were fetched?
      ├── Span: reranking          ← how were they reordered?
      └── Generation: llm-answer   ← tokens, model, cost
```

In [1]:
from langfuse import propagate_attributes
from langfuse_demo.client import get_client
from langfuse_demo.config import settings
from langfuse_demo.llm import call_llm
from langfuse_demo.rag import retrieve, rerank, format_context, DOCS

from dotenv import load_dotenv

_ = load_dotenv()

langfuse = get_client()
print(f"Langfuse connected: {settings.langfuse_host}")

Langfuse connected: http://localhost:3000


---

## The Knowledge Base

Our "internal documents" are 6 in-memory articles (in production: pgvector, Chroma, Pinecone, etc.):

In [2]:
print(f"Knowledge base: {len(DOCS)} documents\n")
for doc in DOCS.values():
    print(f"  [{doc['id']}] {doc['title']}")

Knowledge base: 6 documents

  [onboarding] Onboarding Guide — Development Environment
  [arquitetura] System Architecture
  [api_llm] LLM Inference API
  [mlops] MLOps Pipeline — Training and Deploy
  [langfuse_guia] Observability Guide with Langfuse
  [dados_governanca] Data Governance and Privacy


---

## Testing Retrieval (without trace)

First, see how retrieval works in isolation:

In [3]:
query = "How do I set up the development environment?"
docs = retrieve(query, top_k=3)
docs = rerank(query, docs)

print(f"Query: '{query}'\n")
print("Retrieved documents (order: most relevant first):")
for i, doc in enumerate(docs, 1):
    print(f"{i}. [{doc['id']}] {doc['title']}")
    print(f"BM25 score: {doc['retrieval_score']} | Final score: {doc['final_score']}")

Query: 'How do I set up the development environment?'

Retrieved documents (order: most relevant first):
1. [onboarding] Onboarding Guide — Development Environment
BM25 score: 11.1787 | Final score: 11.7787
2. [langfuse_guia] Observability Guide with Langfuse
BM25 score: 4.0552 | Final score: 4.0552
3. [arquitetura] System Architecture
BM25 score: 2.2221 | Final score: 2.2221


---

## The Full RAG Pipeline with Tracing

Now the same pipeline, but with **span hierarchy in Langfuse**.
Each step becomes an observation with its own metadata.

In [4]:
def rag_pipeline(
    question: str,
    user_id: str = "anonymous",
    session_id: str | None = None,
    top_k: int = 3,
) -> dict:
    """Full RAG pipeline with hierarchical tracing in Langfuse."""

    # propagate_attributes applies user_id/session_id to the entire trace
    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        tags=["rag", "internal-docs", "demo-ch02"],
    ):
        with langfuse.start_as_current_observation(
            as_type="span",
            name="rag-pipeline",
            input={"question": question},
            metadata={"top_k": top_k, "kb_size": len(DOCS)},
        ) as pipeline:
            trace_id = langfuse.get_current_trace_id()

            # ── Step 1: Retrieval ──────────────────────────────────────
            with langfuse.start_as_current_observation(
                as_type="span",
                name="retrieval",
                input={"query": question, "top_k": top_k},
            ) as retrieval_span:
                retrieved = retrieve(question, top_k=top_k)
                retrieval_span.update(
                    output={"docs": [{"id": d["id"], "score": d["retrieval_score"]} for d in retrieved]},
                    metadata={"retriever": "bm25-inMemory"},
                )

            # ── Step 2: Reranking ──────────────────────────────────────
            with langfuse.start_as_current_observation(
                as_type="span",
                name="reranking",
                input={"n_docs": len(retrieved)},
            ) as rerank_span:
                reranked = rerank(question, retrieved)
                rerank_span.update(
                    output={"top_doc": reranked[0]["title"], "final_scores": [d["final_score"] for d in reranked]},
                    metadata={"reranker": "title-overlap-boost"},
                )

            # ── Step 3: Generation ────────────────────────────────────
            context = format_context(reranked)
            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a technical assistant for the Data Science team. "
                        "Answer using ONLY the provided context. "
                        "If the answer is not in the context, say 'I could not find that information in the documents.'"
                    ),
                },
                {
                    "role": "user",
                    "content": f"Context:\n{context}\n\nQuestion: {question}",
                },
            ]

            with langfuse.start_as_current_observation(
                as_type="generation",
                name="llm-answer",
                input=messages,
                metadata={"context_docs": [d["id"] for d in reranked]},
            ) as gen:
                answer, usage = call_llm(messages)
                gen.update(
                    model=settings.groq_model,
                    output=answer,
                    usage_details={"input": usage["input"], "output": usage["output"]},
                    cost_details={"input": usage["input_cost"], "output": usage["output_cost"]},
                )

            pipeline.update(output={"answer": answer})

    return {
        "trace_id": trace_id,
        "answer": answer,
        "docs_used": [d["id"] for d in reranked],
        "tokens": usage,
    }

### Running real questions

Run the cells below and **watch traces appear in the UI in real time**.

In [5]:
result = rag_pipeline(
    question="How do I configure the local development environment?",
    user_id="ana.silva",
    session_id="onboarding-jun-2024",
)
langfuse.flush()

print(f"Answer:\n{result['answer']}")
print(f"\nDocs used: {result['docs_used']}")
print(f"Tokens: input={result['tokens']['input']} | output={result['tokens']['output']}")
print(f"\nTrace: {settings.langfuse_host}/trace/{result['trace_id']}")

Answer:
To configure the local development environment, follow these steps:

1. Ensure you have the prerequisites: Python 3.12+, uv, and Docker Desktop.
2. Clone the repository: git clone https://github.com/celfocus/projeto.git
3. Install dependencies: uv sync
4. Configure environment variables: copy .env.example to .env and fill in the keys
5. Start the local infrastructure: docker compose up -d
6. Verify the setup: uv run python -m pytest tests/

Docs used: ['onboarding', 'langfuse_guia', 'arquitetura']
Tokens: input=406 | output=106

Trace: http://localhost:3000/trace/e2baf8fed9278b2acb1bf7de1833fe76


In [6]:
result2 = rag_pipeline(
    question="Which databases does the system use and what is each one for?",
    user_id="joao.costa",
    session_id="architecture-review-2024",
)
langfuse.flush()

print(f"Answer:\n{result2['answer']}")
print(f"\nTrace: {settings.langfuse_host}/trace/{result2['trace_id']}")

Answer:
The system uses the following databases:

1. **PostgreSQL 16**: for transactional and relational data
2. **ClickHouse**: for analytics and high-volume logs (billions of events)
3. **Redis**: for L2 cache and background job queues, and also for asynchronous service communication through Redis Streams.

Trace: http://localhost:3000/trace/70a4f1f9a38132223e7fbf8260d5244d


In [7]:
# Question whose context is NOT in the docs — the model should admit it doesn't know
result3 = rag_pipeline(
    question="What is the onboarding process for new external clients?",
    user_id="carla.mendes",
)
langfuse.flush()

print(f"Answer:\n{result3['answer']}")
print(f"\nTrace: {settings.langfuse_host}/trace/{result3['trace_id']}")
print("\nIn the UI: see that retrieval returned irrelevant docs → model admitted it didn't know.")

Answer:
I could not find that information in the documents.

Trace: http://localhost:3000/trace/7f4912713d3faa66c8b58281b7903ccf

In the UI: see that retrieval returned irrelevant docs → model admitted it didn't know.


---

## What to explore in the UI

Open any of the 3 traces above and see:

1. **Visual timeline**: retrieval → reranking → llm-answer in cascade
2. **Retrieval span**: which docs were fetched and with what score
3. **Reranking span**: how the order changed after reranking
4. **Generation**: input/output tokens, model used, LLM call latency
5. **Metadata**: filter traces by `user_id`, `session_id`, `tags`

**Real debugging scenario:**  
If a user reports a wrong answer → open the trace → check if retrieval got the right docs → if not, the problem is in the retriever, not the LLM.

---

## Chapter 2 Summary

| Concept | Implementation |
|----------|---------------|
| Span hierarchy | Nested `start_as_current_observation` |
| Metadata per step | `span.update(output=..., metadata=...)` |
| Tokens in generation | `gen.update(usage_details={"input": ..., "output": ...})` |
| User and session tracking | `propagate_attributes(user_id=..., session_id=...)` |

**Next chapter**: Prompt Management + versioning + multi-turn sessions